# 02 - Feature Engineering

Extract aggregate features from trajectory data for behavior analysis.

## Steps:
1. Load cleaned trajectory data
2. Extract trip-level features
3. Standardize features
4. Save feature matrix


In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd

from src.feature_engineering import (
    extract_trip_features,
    filter_trips,
    select_features_for_clustering,
    standardize_features,
)

# Load cleaned data
data_dir = Path("../data/processed")
df = pd.read_parquet(data_dir / "trajectories_cleaned.parquet")

print(f"Loaded {len(df)} GPS points from {df['trajectory_id'].nunique()} trajectories")


Loaded 2396 GPS points from 2 trajectories


## Extract Features

Compute aggregate features for each trip.


In [3]:
# Extract trip features
features_df = extract_trip_features(df)

print(f"Extracted features for {len(features_df)} trips")
print("\nFeature columns:")
print(features_df.columns.tolist())

features_df.head()


Extracted features for 2 trips

Feature columns:
['trajectory_id', 'user_id', 'n_points', 'trip_duration_min', 'trip_distance_km', 'mean_speed_kmh', 'std_speed_kmh', 'max_speed_kmh', 'speed_cv', 'avg_speed_distance_kmh', 'mean_accel_ms2', 'std_accel_ms2', 'max_accel_ms2', 'min_accel_ms2', 'pct_time_stopped', 'mean_jerk', 'std_jerk']


,trajectory_id,user_id,n_points,trip_duration_min,trip_distance_km,mean_speed_kmh,std_speed_kmh,max_speed_kmh,speed_cv,avg_speed_distance_kmh,mean_accel_ms2,std_accel_ms2,max_accel_ms2,min_accel_ms2,pct_time_stopped,mean_jerk,std_jerk
0,20090102043127,Data,60,8.966667,0.259307,3.876083,13.664483,107.060811,3.525333,1.735137,-0.114333,0.806871,0.668551,-5.779428,26.666667,0.422788,1.088810
1,20090103012134,Data,2336,257.950000,8.793159,3.941037,4.476762,86.438037,1.135935,2.045317,0.042131,0.594292,8.809978,-9.010207,26.926370,0.318124,0.893033


## Filter Short Trips

Remove trips that are too short to be meaningful.


In [4]:
# Filter short trips
features_df = filter_trips(features_df, min_duration_min=1.0, min_distance_km=0.1)

print(f"After filtering: {len(features_df)} trips")
print("\nTrip statistics:")
print(features_df[['trip_duration_min', 'trip_distance_km']].describe())


After filtering: 2 trips

Trip statistics:
       trip_duration_min  trip_distance_km
count           2.000000          2.000000
mean          133.458333          4.526233
std           176.057803          6.034345
min             8.966667          0.259307
25%            71.212500          2.392770
50%           133.458333          4.526233
75%           195.704167          6.659696
max           257.950000          8.793159


## Standardize Features

Standardize features for clustering.


In [5]:
# Standardize features
features_scaled = standardize_features(features_df)

# Select features for clustering
feature_cols = select_features_for_clustering(features_scaled)
print(f"Selected {len(feature_cols)} features for clustering:")
print(feature_cols)

features_scaled.head()


Selected 7 features for clustering:
['mean_speed_kmh_std', 'std_speed_kmh_std', 'max_speed_kmh_std', 'mean_accel_ms2_std', 'std_accel_ms2_std', 'pct_time_stopped_std', 'mean_jerk_std']


,trajectory_id,user_id,n_points,trip_duration_min,trip_distance_km,mean_speed_kmh,std_speed_kmh,max_speed_kmh,speed_cv,avg_speed_distance_kmh,...,max_speed_kmh_std,speed_cv_std,avg_speed_distance_kmh_std,mean_accel_ms2_std,std_accel_ms2_std,max_accel_ms2_std,min_accel_ms2_std,pct_time_stopped_std,mean_jerk_std,std_jerk_std
0,20090102043127,Data,60,8.966667,0.259307,3.876083,13.664483,107.060811,3.525333,1.735137,...,1.0,1.0,-1.0,-1.0,1.0,-1.0,1.0,-1.0,1.0,1.0
1,20090103012134,Data,2336,257.950000,8.793159,3.941037,4.476762,86.438037,1.135935,2.045317,...,-1.0,-1.0,1.0,1.0,-1.0,1.0,-1.0,1.0,-1.0,-1.0


## Save Features

Save feature matrix for clustering.


In [6]:
# Save features
output_file = data_dir / "trip_features.csv"
features_scaled.to_csv(output_file, index=False)
print(f"Saved features to {output_file}")

# Also save with standardized features for clustering
feature_df_for_clustering = features_scaled[["trajectory_id"] + feature_cols]
cluster_file = data_dir / "trip_features_clustering.csv"
feature_df_for_clustering.to_csv(cluster_file, index=False)
print(f"Saved clustering features to {cluster_file}")


Saved features to ../data/processed/trip_features.csv
Saved clustering features to ../data/processed/trip_features_clustering.csv
